# Остывание спутника: что делать, если уравнение не решается

**Задача.** Спутник нагревается Солнцем и остывает излучением:

$$\frac{dT}{dt} = Q_0 + A\cos\omega t - \sigma T^4, \qquad Q_0 = A = \sigma = \omega = 1.$$

В элементарных функциях это уравнение не решается. Но узнать про решение можно почти всё — не решая.

**Как запустить.** Нажмите **Ctrl+F9** (или «Среда выполнения» → «Выполнить всё»). На предупреждение «блокнот создан не Google» ответьте «Всё равно выполнить». Через несколько секунд ниже появится картинка с кнопками и ползунками.

**Код спрятан, его можно открыть и просмотреть.** Всё управление — кнопками и ползунками под заголовками.

In [ ]:
#@title Движок — можно открыть { display-mode: "form" }
# =============================================================
#  СПУТНИК — движок: уравнение, поле, изоклины, Эйлер, Пикар.
#  Интерфейс — в следующей ячейке. Здесь ничего не рисуется.
# =============================================================
import warnings
import numpy as np
import sympy as sp
import contourpy
from scipy.integrate import solve_ivp, cumulative_trapezoid

warnings.filterwarnings('ignore', category=RuntimeWarning)

# Числа с пары: Q0 = A = sigma = omega = 1. В виджете меняется только A,
# и только в пределах 0 <= A <= Q0 (нагрев не бывает отрицательным).
Q0, SIGMA, OMEGA = 1.0, 1.0, 1.0
BIG = 1e6                     # |T| больше этого — считаем, что Эйлер разлетелся


class Equation:
    """Правая часть y' = f(t, y) плюс всё, что про неё нужно знать виджету."""

    def __init__(self, f, f_y, latex, name='y', physical_positive=False,
                 equilibrium=None):
        self.f = f                      # f(t, y), векторизована
        self.f_y = f_y                  # df/dy(t, y) — для оценки шага
        self.latex = latex
        self.name = name                # как подписывать ось: 'T' или 'y'
        self.physical_positive = physical_positive   # T < 0 не бывает
        self.equilibrium = equilibrium  # кривая f = 0 (если знаем формулой)


def sputnik(A):
    """T' = Q0 + A cos(wt) - sigma T^4 — уравнение с пары."""
    def f(t, T):
        return Q0 + A * np.cos(OMEGA * t) - SIGMA * T**4

    def f_y(t, T):
        return -4.0 * SIGMA * T**3

    def eq_curve(t):
        # изоклина m = 0: здесь у решений экстремумы
        return np.maximum(Q0 + A * np.cos(OMEGA * t), 0.0) ** 0.25

    a = ('%.2f' % A).rstrip('0').rstrip('.')
    latex = r"T' = 1 + %s\cos t - T^{4}" % ('' if A == 1 else a)
    if A == 0:
        latex = r"T' = 1 - T^{4}"
    return Equation(f, f_y, latex, name='T', physical_positive=True,
                    equilibrium=eq_curve)


_t, _y = sp.symbols('t y')


def parse_equation(text):
    """Строка вида '5 - 3*sqrt(y)' -> Equation. Переменные: t и y (T = y)."""
    from sympy.parsing.sympy_parser import (
        parse_expr, standard_transformations, convert_xor,
        implicit_multiplication_application)
    tr = standard_transformations + (implicit_multiplication_application,
                                     convert_xor)
    expr = parse_expr(text, transformations=tr,
                      local_dict={'t': _t, 'y': _y, 'T': _y, 'x': _t,
                                  'e': sp.E, 'pi': sp.pi})
    extra = expr.free_symbols - {_t, _y}
    if extra:
        raise ValueError('непонятные буквы: %s (можно только t и y)'
                         % ', '.join(sorted(map(str, extra))))
    fn = sp.lambdify((_t, _y), expr, 'numpy')
    dfn = sp.lambdify((_t, _y), sp.diff(expr, _y), 'numpy')

    def wrap(g):
        def h(t, y):
            with np.errstate(all='ignore'):
                v = np.asarray(g(t, y), dtype=complex)
            v = np.where(np.abs(v.imag) > 1e-12, np.nan, v.real)
            return np.broadcast_to(v, np.broadcast(t, y).shape).astype(float)
        return h

    return Equation(wrap(fn), wrap(dfn), "y' = " + sp.latex(expr), name='y')


# ---------------- точное (эталонное) решение ----------------
def reference(eq, y0, t_end, t0=0.0):
    """Жёсткий решатель Radau: он не боится T^4. Возвращает dense-функцию
    и момент, до которого решение дожило (если улетело — раньше t_end)."""
    def rhs(t, Y):
        return [float(eq.f(t, Y[0]))]

    def blowup(t, Y):
        return abs(Y[0]) - 1e3
    blowup.terminal = True
    try:
        s = solve_ivp(rhs, [t0, t_end], [y0], method='Radau', rtol=1e-9,
                      atol=1e-11, dense_output=True, events=blowup)
    except Exception:
        return None, t0
    if s.sol is None or s.t.size < 2:
        return None, t0
    return s.sol, float(s.t[-1])


# ---------------- метод Эйлера ----------------
def euler(eq, y0, dt, t_end, t0=0.0):
    n = int(np.floor((t_end - t0) / dt + 1e-9))
    t = t0 + dt * np.arange(n + 1)
    y = np.full(n + 1, np.nan)
    y[0] = y0
    for k in range(n):
        v = y[k] + dt * float(eq.f(t[k], y[k]))
        if not np.isfinite(v) or abs(v) > BIG:
            break
        y[k + 1] = v
    return t, y


def euler_error_sweep(eq, y0, t_err, dts):
    """Ошибка Эйлера в зависимости от шага: наибольшая на [0, t_err] и в конце.
    Флаги: ушёл ли в T < 0 (для спутника) и разлетелся ли совсем."""
    sol, t_alive = reference(eq, y0, t_err)
    out = dict(dt=np.asarray(dts), sup=np.full(len(dts), np.nan),
               end=np.full(len(dts), np.nan),
               neg=np.zeros(len(dts), bool), diverged=np.zeros(len(dts), bool))
    if sol is None or t_alive < t_err - 1e-9:
        return out
    for i, dt in enumerate(dts):
        t, y = euler(eq, y0, dt, t_err)
        if not np.all(np.isfinite(y)):
            out['diverged'][i] = True
            continue
        e = np.abs(y - sol(t)[0])
        out['sup'][i] = e.max()
        out['end'][i] = e[-1]
        out['neg'][i] = eq.physical_positive and y.min() < 0
    return out


# ---------------- поле направлений и изоклины ----------------
def slope_segments(t, y, m, sx, sy, length_px):
    """Короткие отрезки наклона m в точках (t, y) одинаковой ЭКРАННОЙ длины:
    sx, sy — пикселей на единицу по осям. Без этого при наклоне -250 чёрточки
    были бы то точками, то заборами."""
    t, y, m = np.broadcast_arrays(np.asarray(t, float), np.asarray(y, float),
                                  np.asarray(m, float))
    dx, dy = sx * np.ones_like(m), sy * m
    nrm = np.hypot(dx, dy)
    ux, uy = dx / nrm, dy / nrm
    ht = 0.5 * length_px * ux / sx
    hy = 0.5 * length_px * uy / sy
    seg = np.stack([np.stack([t - ht, y - hy], -1),
                    np.stack([t + ht, y + hy], -1)], 1)
    ok = np.isfinite(seg).all(axis=(1, 2))
    return seg[ok]


def isoclines(eq, levels, t_rng, y_rng, n=(500, 360)):
    """Изоклина m — линия уровня f(t, y) = m. Считается честно по сетке,
    поэтому работает для любого уравнения, а не только для спутника."""
    tg = np.linspace(*t_rng, n[0])
    yg = np.linspace(*y_rng, n[1])
    Tg, Yg = np.meshgrid(tg, yg)
    Z = np.ma.masked_invalid(eq.f(Tg, Yg))
    gen = contourpy.contour_generator(tg, yg, Z)
    return {m: [ln for ln in gen.lines(m) if len(ln) > 1] for m in levels}


def points_along(lines, k):
    """k точек, равномерно по длине, вдоль набора ломаных — чтобы расставить
    на изоклине чёрточки, как на доске."""
    pts = []
    total = sum(np.hypot(*np.diff(ln, axis=0).T).sum() for ln in lines)
    if total == 0:
        return np.empty((0, 2))
    for ln in lines:
        d = np.r_[0, np.cumsum(np.hypot(*np.diff(ln, axis=0).T))]
        kk = max(1, int(round(k * d[-1] / total)))
        s = (np.arange(kk) + 0.5) * d[-1] / kk
        pts.append(np.c_[np.interp(s, d, ln[:, 0]), np.interp(s, d, ln[:, 1])])
    return np.vstack(pts)


# ---------------- метод Пикара ----------------
def picard(eq, y0, t_grid, n_iter, cap=1e3):
    """Итерации y_{n+1}(t) = y0 + int_0^t f(s, y_n(s)) ds на сетке.
    Где итерация ушла за |y| > cap, дальше она для картинки бессмысленна."""
    out = [np.full_like(t_grid, y0, dtype=float)]
    for _ in range(n_iter):
        prev = out[-1]
        integrand = eq.f(t_grid, np.where(np.abs(prev) > cap, np.nan, prev))
        nxt = y0 + cumulative_trapezoid(integrand, t_grid, initial=0.0)
        nxt[np.abs(nxt) > cap] = np.nan
        out.append(nxt)
    return out


# ---------------- периодический режим спутника ----------------
def periodic_regime(A, n_settle=8, n=20000):
    """Один период установившегося режима. Решения забывают начальное условие
    за время порядка 1/(4 sigma T^3) ~ 0.25..1, так что 8 периодов — с огромным
    запасом. Отсчёт — от целого числа периодов, чтобы фаза считалась от
    максимума нагрева (cos = 1)."""
    t_settle = n_settle * 2 * np.pi / OMEGA
    eq = sputnik(A)
    s = solve_ivp(lambda t, Y: [eq.f(t, Y[0])], [0, t_settle + 2 * np.pi],
                  [1.0], method='Radau', rtol=1e-10, atol=1e-12,
                  dense_output=True)
    t = np.linspace(t_settle, t_settle + 2 * np.pi, n, endpoint=False)
    T = s.sol(t)[0]
    tt = t - t_settle                     # фаза отсчитана от максимума нагрева
    a1 = 2 * np.mean(T * np.cos(OMEGA * tt))
    b1 = 2 * np.mean(T * np.sin(OMEGA * tt))
    return dict(t=tt, T=T, mean=T.mean(), rad=np.mean(T**4) ** 0.25,
                amp1=np.hypot(a1, b1), phase1=np.degrees(np.arctan2(b1, a1)),
                peak=np.degrees(OMEGA * tt[np.argmax(T)]),
                trough=np.degrees(OMEGA * tt[np.argmin(T)]),
                half=(T.max() - T.min()) / 2, Tmin=T.min(), Tmax=T.max())


def linear_prediction(A):
    """Что даст П4: линеаризация вокруг T_рад = (Q0/sigma)^(1/4) —
    единственной температуры, известной до решения."""
    T_star = (Q0 / SIGMA) ** 0.25
    alpha = 4 * SIGMA * T_star**3
    return dict(T_star=T_star, alpha=alpha,
                amp=A / np.hypot(alpha, OMEGA),
                phase=np.degrees(np.arctan2(OMEGA, alpha)))


print('движок загружен')

In [ ]:
#@title Виджет: изоклины и метод Эйлера { display-mode: "form" }
# =============================================================
#  СПУТНИК — интерфейс: поле направлений, изоклины, Эйлер.
#  Требует предыдущую ячейку (движок). Запускать после неё.
# =============================================================
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.transforms import Bbox, blended_transform_factory

MODE_SAT, MODE_OWN = 'sat', 'own'
SAT_T = (0.0, 4 * np.pi)            # окно для спутника: два периода нагрева
SAT_Y = (-1.6, 4.6)                 # ниже нуля — видно, куда уходит Эйлер
SAT_LEVELS = [1.5, 1.0, 0.0, -1.0, -15.0]   # те же m, что на доске
SAT_FAMILY = [0.3, 1.0, 2.0, 4.0]   # 0,3 — ниже равновесия: подход снизу
OWN_T = (0.0, 6.0)
T_ERR = 2.0                          # на каком отрезке мерить ошибку Эйлера
N_SWEEP = 40                         # точек на графике ошибки
DT_MAX = 0.5                         # самый большой шаг, как у ползунка
CONTINUOUS = True                    # False — если на слабой машине дёргается

_SL = dict(style={'description_width': '110px'},
           layout=widgets.Layout(width='320px'))


def _fmt(x, d=4):
    return ('%.*f' % (d, x)).replace('.', ',')


class SputnikApp:

    def __init__(self):
        self._busy = False
        self._sweep_key, self._sweep = None, None
        self._iso_key, self._iso = None, None
        self._fam_key, self._fam = None, None
        self._build_controls()
        self._build_figure()
        self._wire()
        display(self.panel, self.out, self.info)
        self._refresh()

    # ---------------- органы управления ----------------
    def _build_controls(self):
        self.w_mode = widgets.ToggleButtons(
            options=[('спутник с пары', MODE_SAT), ('своё уравнение', MODE_OWN)],
            value=MODE_SAT, description='Уравнение:',
            style={'description_width': '85px', 'button_width': '150px'})
        self.w_text = widgets.Text(value='y^2', description="y' =",
                                   continuous_update=False, disabled=True,
                                   style={'description_width': '40px'},
                                   layout=widgets.Layout(width='330px'))
        self.w_A = widgets.FloatSlider(value=1.0, min=0.0, max=1.0, step=0.05,
                                       readout_format='.2f', description='нагрев A:',
                                       continuous_update=False, **_SL)
        self.w_y0 = widgets.FloatSlider(value=2.0, min=0.0, max=4.5, step=0.05,
                                        readout_format='.2f',
                                        description='начало T(0):',
                                        continuous_update=False, **_SL)
        self.w_dt = widgets.FloatLogSlider(value=0.01, base=10, min=-3,
                                           max=np.log10(0.5), step=0.02,
                                           readout_format='.4f',
                                           description='шаг Эйлера Δt:',
                                           continuous_update=CONTINUOUS, **_SL)
        self.w_m = widgets.FloatSlider(value=1.0, min=-20.0, max=2.0, step=0.5,
                                       readout_format='.1f',
                                       description='изоклина m:',
                                       continuous_update=CONTINUOUS, **_SL)
        self.w_field = widgets.Checkbox(value=False, indent=False,
                                        description='поле направлений')
        self.w_iso = widgets.Checkbox(value=True, indent=False,
                                      description='изоклины')
        self.w_fam = widgets.Checkbox(value=True, indent=False,
                                      description='решения')
        self.w_eu = widgets.Checkbox(value=False, indent=False,
                                     description='Эйлер')
        self.b_board = widgets.Button(description='Изоклины как на доске',
                                      layout=widgets.Layout(width='185px'))
        self.b_eu_ok = widgets.Button(description='Эйлер с доски',
                                      layout=widgets.Layout(width='130px'))
        self.b_eu_bad = widgets.Button(description='Эйлер врёт',
                                       layout=widgets.Layout(width='115px'))
        self.b_reset = widgets.Button(description='Сброс',
                                      layout=widgets.Layout(width='80px'))
        self.out = widgets.Output()
        self.info = widgets.HTML()
        self.panel = widgets.VBox([
            widgets.HTML("<h3 style='margin:2px 0'>Остывание спутника: "
                         "узнать решение, не решая</h3>"),
            widgets.HBox([self.w_mode, self.w_text]),
            widgets.HBox([self.w_A, self.w_y0, self.w_dt]),
            widgets.HBox([self.w_m, self.w_field, self.w_iso, self.w_fam,
                          self.w_eu]),
            widgets.HBox([self.b_board, self.b_eu_ok, self.b_eu_bad,
                          self.b_reset]),
        ])

    def _wire(self):
        for w in (self.w_text, self.w_A, self.w_y0, self.w_dt,
                  self.w_m, self.w_field, self.w_iso, self.w_fam, self.w_eu):
            w.observe(self._on_change, names='value')
        self.w_mode.observe(self._on_mode, names='value')
        self.b_board.on_click(lambda _b: self._set(
            w_mode=MODE_SAT, w_A=1.0, w_y0=2.0, w_m=1.0, w_field=False,
            w_iso=True, w_fam=True, w_eu=False))
        self.b_eu_ok.on_click(lambda _b: self._set(
            w_mode=MODE_SAT, w_A=1.0, w_y0=2.0, w_dt=0.01, w_iso=False,
            w_field=True, w_fam=True, w_eu=True))
        self.b_eu_bad.on_click(lambda _b: self._set(
            w_mode=MODE_SAT, w_A=1.0, w_y0=4.0, w_dt=0.02, w_iso=False,
            w_field=True, w_fam=True, w_eu=True))
        self.b_reset.on_click(lambda _b: self._set(
            w_mode=MODE_SAT, w_A=1.0, w_y0=2.0, w_dt=0.01, w_m=1.0,
            w_field=False, w_iso=True, w_fam=True, w_eu=False))

    def _on_mode(self, change):
        # у своего уравнения другие разумные пределы; сдвиг пределов может
        # сдвинуть значения ползунков — перерисовываем один раз, в конце
        prev, self._busy = self._busy, True
        try:
            if change['new'] == MODE_SAT:
                self.w_y0.min, self.w_y0.max = 0.0, 4.5
                self.w_y0.description = 'начало T(0):'
                self.w_m.min, self.w_m.max = -20.0, 2.0
            else:
                self.w_y0.max, self.w_y0.min = 5.0, -5.0
                self.w_y0.value = 1.0
                self.w_y0.description = 'начало y(0):'
                self.w_m.min, self.w_m.max = -20.0, 20.0
                self.w_m.value = 0.0
            self.w_A.disabled = change['new'] != MODE_SAT
            self.w_text.disabled = change['new'] == MODE_SAT
        finally:
            self._busy = prev
        if not prev:
            self._refresh()

    # ---------------- фигура строится ОДИН раз ----------------
    def _build_figure(self):
        self.fig, (self.ax, self.ax2) = plt.subplots(
            1, 2, figsize=(11.2, 4.9), dpi=88,
            gridspec_kw={'width_ratios': [2.6, 1.0]})
        self.fig.subplots_adjust(left=0.055, right=0.985, top=0.90,
                                 bottom=0.115, wspace=0.26)
        plt.close(self.fig)

        ax = self.ax
        self.a_neg = ax.axhspan(-100, 0, color='#f3dede', alpha=0.7, lw=0,
                                zorder=0)
        self.a_neg_txt = ax.text(0.99, 0.03, 'T < 0: так не бывает',
                                 transform=ax.transAxes, ha='right',
                                 color='#b04040', fontsize=10, zorder=10)
        self.a_field = LineCollection([], colors='#b0b0b0', linewidths=1.0,
                                      zorder=1)
        ax.add_collection(self.a_field)
        self.a_iso = LineCollection([], colors='#8c6fc4', linewidths=1.0,
                                    linestyles=':', zorder=2)
        ax.add_collection(self.a_iso)
        self.a_iso_hi = LineCollection([], colors='#6f4fb0', linewidths=1.6,
                                       zorder=3)
        ax.add_collection(self.a_iso_hi)
        self.a_iso_dash = LineCollection([], colors='#e0301e', linewidths=2.0,
                                         zorder=4)
        ax.add_collection(self.a_iso_dash)
        self.a_fam = LineCollection([], colors='#7a9cc6', linewidths=1.2,
                                    zorder=5)
        ax.add_collection(self.a_fam)
        self.a_ref, = ax.plot([], [], color='#1f4e9c', lw=2.4, zorder=6,
                              label='точное решение')
        self.a_eu, = ax.plot([], [], color='#d62728', lw=1.4, marker='o',
                             ms=3.2, zorder=7, label='Эйлер')
        self.a_start, = ax.plot([], [], 'o', color='black', ms=6, zorder=8)
        self.a_iso_txt = ax.text(0.01, 0.985, '', transform=ax.transAxes,
                                 ha='left', va='top', fontsize=9,
                                 color='#6f4fb0', zorder=10,
                                 bbox=dict(fc='white', ec='none', alpha=0.8))
        self.a_hi_txt = ax.text(0, 0, '', fontsize=10, color='#6f4fb0',
                                ha='center', va='bottom', zorder=10,
                                fontweight='bold')
        ax.axhline(0, color='#888888', lw=0.8, zorder=1)
        ax.set_xlabel('t')
        ax.grid(True, alpha=0.25)

        # правая панель видна только вместе с Эйлером; без неё поле
        # направлений занимает всю ширину
        self._pos_split = self.ax.get_position().frozen()
        p2 = self.ax2.get_position()
        self._pos_full = Bbox.from_extents(self._pos_split.x0, self._pos_split.y0,
                                           p2.x1, self._pos_split.y1)

        ax2 = self.ax2
        self.b_sup, = ax2.plot([], [], '-o', color='#d62728', ms=3.5, lw=1.5,
                               label='наибольшая ошибка')
        self.b_guide, = ax2.plot([], [], '--', color='#555555', lw=1.0,
                                 label='пропорционально шагу')
        self.b_top, = ax2.plot([], [], '^', color='#999999', ms=6,
                               label='ушла выше графика')
        self.b_neg, = ax2.plot([], [], '^', color='#ef8a00', ms=7,
                               label='и Эйлер заходил в T < 0')
        self.b_div, = ax2.plot([], [], 'x', color='black', ms=7, mew=1.6,
                               label='Эйлер разлетелся')
        self.b_now = ax2.axvline(0.01, color='#d62728', lw=1.0, alpha=0.6)
        self.b_thr = ax2.axvline(0.01, color='black', lw=1.0, ls='-.')
        tr = blended_transform_factory(ax2.transData, ax2.transAxes)
        self.b_thr_txt = ax2.text(0.01, 0.97, 'граница \nустойчивости ',
                                  transform=tr, fontsize=8.5, va='top',
                                  ha='right')
        self.b_now_txt = ax2.text(0.98, 0.5, 'шаг Δt —\nправее края →',
                                  transform=ax2.transAxes, ha='right',
                                  fontsize=8.5, color='#d62728')
        self.b_msg = ax2.text(0.5, 0.6, '', transform=ax2.transAxes,
                              ha='center', va='center', fontsize=10,
                              color='#777777')
        ax2.set_xlabel('шаг Δt')
        ax2.set_ylabel('наибольшая ошибка на [0, %g]' % T_ERR)
        ax2.set_title('Ошибка Эйлера', fontsize=12)
        ax2.grid(True, alpha=0.25)

    # ---------------- вычисления ----------------
    def _equation(self):
        if self.w_mode.value == MODE_SAT:
            return sputnik(self.w_A.value), None
        try:
            return parse_equation(self.w_text.value), None
        except Exception as e:
            return None, str(e)

    def _window(self, eq, key):
        if self.w_mode.value == MODE_SAT:
            return SAT_T, SAT_Y
        # для своего уравнения окно подбирается по решениям, но не шире ±10
        ys = [self.w_y0.value, -1.0, 1.0]
        for y0 in np.linspace(-3, 3, 7):
            sol, t_alive = reference(eq, y0, OWN_T[1])
            if sol is not None:
                v = sol(np.linspace(0, t_alive, 200))[0]
                ys.extend(v[np.isfinite(v)].tolist())
        lo, hi = np.clip(np.percentile(ys, [3, 97]), -10, 10)
        pad = 0.12 * max(hi - lo, 1.0)
        return OWN_T, (lo - pad, hi + pad)

    def _scales(self, t_rng, y_rng):
        bb = self.ax.get_position()
        w_px = bb.width * self.fig.get_figwidth() * self.fig.dpi
        h_px = bb.height * self.fig.get_figheight() * self.fig.dpi
        return w_px / (t_rng[1] - t_rng[0]), h_px / (y_rng[1] - y_rng[0])

    # ---------------- отрисовка ----------------
    def render(self):
        eq, err = self._equation()
        mode = self.w_mode.value
        if eq is None:
            self.ax.set_title('Не получилось прочитать уравнение: %s' % err,
                              color='#b00000', fontsize=11)
            return ("<div style='color:#b00000'>Пишите через t и y, например "
                    "<code>5 - 3*sqrt(y)</code> или <code>t - y^2</code>.</div>")

        key = (mode, self.w_text.value if mode == MODE_OWN else self.w_A.value)
        if key != getattr(self, '_win_key', None):
            self._win = self._window(eq, key)
            self._win_key = key
        t_rng, y_rng = self._win
        show_err = self.w_eu.value
        self.ax2.set_visible(show_err)
        self.ax.set_position(self._pos_split if show_err else self._pos_full)
        sx, sy = self._scales(t_rng, y_rng)
        ax, y0, dt, m = self.ax, self.w_y0.value, self.w_dt.value, self.w_m.value
        nm = eq.name

        # --- поле направлений ---
        if self.w_field.value:
            tg, yg = np.meshgrid(np.linspace(*t_rng, 27)[1:-1],
                                 np.linspace(*y_rng, 17)[1:-1])
            self.a_field.set_segments(
                slope_segments(tg.ravel(), yg.ravel(),
                               eq.f(tg.ravel(), yg.ravel()), sx, sy, 13))
        else:
            self.a_field.set_segments([])

        # --- изоклины: семейство (пунктир) и одна выделенная с чёрточками ---
        if mode == MODE_SAT:
            levels = SAT_LEVELS
        else:
            tg, yg = np.meshgrid(np.linspace(*t_rng, 60), np.linspace(*y_rng, 60))
            z = eq.f(tg, yg)
            z = z[np.isfinite(z)]
            levels = sorted(set(float('%.1g' % v) if abs(v) < 1 else round(v, 1)
                                for v in np.percentile(z, [10, 30, 50, 70, 90]))) \
                if z.size else []
        iso_key = (key, tuple(levels), t_rng, y_rng)
        if iso_key != self._iso_key:
            self._iso = isoclines(eq, levels, t_rng, y_rng)
            self._iso_key = iso_key
        if self.w_iso.value:
            segs = [ln for lv in levels for ln in self._iso[lv]]
            self.a_iso.set_segments(segs)
            lv_txt = '; '.join(('%g' % lv).replace('.', ',') for lv in levels)
            self.a_iso_txt.set_text(
                'пунктир — изоклины m = %s%s\nсплошная с красными чёрточками — m = %s'
                % (lv_txt, ' (снизу вверх)' if mode == MODE_SAT else '',
                   ('%g' % m).replace('.', ',')))
            hi = isoclines(eq, [m], t_rng, y_rng)[m]
            self.a_iso_hi.set_segments(hi)
            pts = points_along(hi, 16)
            self.a_iso_dash.set_segments(
                slope_segments(pts[:, 0], pts[:, 1], m, sx, sy, 20)
                if len(pts) else [])
            # подпись выделенной изоклины — у её самой высокой точки,
            # на физической ветви
            allp = np.vstack(hi) if hi else np.empty((0, 2))
            if eq.physical_positive:
                allp = allp[allp[:, 1] > 0]
            allp = allp[(allp[:, 1] < y_rng[1]) & (allp[:, 1] > y_rng[0])]
            if len(allp):
                p = allp[np.argmax(allp[:, 1])]
                w = t_rng[1] - t_rng[0]
                p0 = min(max(p[0], t_rng[0] + 0.06 * w), t_rng[1] - 0.06 * w)
                self.a_hi_txt.set_position(
                    (p0, p[1] + 0.02 * (y_rng[1] - y_rng[0])))
                self.a_hi_txt.set_text('')
            else:
                self.a_hi_txt.set_text('')
        else:
            for a in (self.a_iso, self.a_iso_hi, self.a_iso_dash):
                a.set_segments([])
            self.a_iso_txt.set_text('')
            self.a_hi_txt.set_text('')

        # --- решения ---
        fam_key = (key, t_rng, y_rng)
        if fam_key != self._fam_key:
            starts = SAT_FAMILY if mode == MODE_SAT else \
                list(np.linspace(y_rng[0], y_rng[1], 8)[1:-1])
            fam = []
            for s0 in starts:
                sol, t_alive = reference(eq, s0, t_rng[1])
                if sol is not None:
                    tt = np.linspace(0, t_alive, 600)
                    fam.append(np.c_[tt, sol(tt)[0]])
            self._fam, self._fam_key = fam, fam_key
        self.a_fam.set_segments(self._fam if self.w_fam.value else [])

        sol, t_alive = reference(eq, y0, t_rng[1])
        if sol is not None:
            tt = np.linspace(0, t_alive, 1500)
            self.a_ref.set_data(tt, sol(tt)[0])
        else:
            self.a_ref.set_data([], [])
        self.a_start.set_data([0.0], [y0])

        # --- Эйлер ---
        te, ye = euler(eq, y0, dt, t_rng[1])
        if self.w_eu.value:
            self.a_eu.set_data(te, ye)
            self.a_eu.set_marker('o' if len(te) <= 260 else 'None')
        else:
            self.a_eu.set_data([], [])
        ax.legend(handles=[self.a_ref] + ([self.a_eu] if self.w_eu.value else []),
                  loc='upper right', fontsize=9, framealpha=0.9)

        # граница 2/|df/dy| имеет смысл, только если решения рядом сходятся
        # (df/dy < 0): при df/dy > 0 растёт и само решение, порога нет
        fy = float(eq.f_y(0.0, y0))
        thr = 2.0 / -fy if fy < -1e-12 else np.inf
        if show_err:
            self._draw_error(eq, key, y0, dt, thr)

        # --- оси и заголовок ---
        ax.set_xlim(*t_rng)
        ax.set_ylim(*y_rng)
        ax.set_ylabel(nm)
        self.a_neg.set_visible(eq.physical_positive)
        self.a_neg_txt.set_visible(eq.physical_positive)
        if mode == MODE_SAT:
            ax.set_title('$%s$' % eq.latex, fontsize=13, pad=8, color='black')
            ax.set_xticks(np.arange(0, 4 * np.pi + 0.1, np.pi / 2))
            ax.set_xticklabels(['0', 'π/2', 'π', '3π/2', '2π', '5π/2', '3π',
                                '7π/2', '4π'])
        else:
            ax.set_title("y' = %s" % self.w_text.value, fontsize=13, pad=8,
                         color='black')
            ax.set_xticks(np.arange(0, OWN_T[1] + 0.1, 1.0))
            ax.set_xticklabels([str(int(v)) for v in np.arange(0, OWN_T[1] + 0.1, 1.0)])

        return self._info_html(eq, y0, dt, m, te, ye, sol, t_alive, thr)

    # ---------------- график ошибки от шага ----------------
    def _draw_error(self, eq, key, y0, dt, thr):
        # шаги — от нуля до трёх границ устойчивости: тогда на малых шагах
        # видна прямая через ноль, а за границей — стена
        x_max = min(DT_MAX, 3 * thr) if np.isfinite(thr) else DT_MAX
        sw_key = (key, y0)
        if sw_key != self._sweep_key:          # дорого — только при смене
            dts = np.linspace(x_max / N_SWEEP, x_max, N_SWEEP)
            self._sweep = euler_error_sweep(eq, y0, T_ERR, dts)
            self._sweep_key = sw_key
        sw, ax2 = self._sweep, self.ax2
        d, e = sw['dt'], sw['sup']
        ok = np.isfinite(e)

        # верх графика — по участку до полутора границ: видно и то, что до
        # границы, и то, что за ней ошибка продолжает расти; обрезка масштабом
        # не должна сама рисовать «стену» ровно на границе
        base = ok & (d <= min(1.5 * thr, x_max) + 1e-12)
        ref = e[base] if base.any() else e[ok]
        y_top = 1.15 * ref.max() if ref.size and ref.max() > 0 else 1.0
        above = ok & (e > y_top)
        self.b_sup.set_data(d, np.where(ok & ~above, e, np.nan))
        self.b_top.set_data(d[above & ~sw['neg']],
                            np.full((above & ~sw['neg']).sum(), 0.96 * y_top))
        neg_show = sw['neg'] & ok
        self.b_neg.set_data(d[neg_show],
                            np.where(above[neg_show], 0.96 * y_top, e[neg_show]))
        self.b_div.set_data(d[sw['diverged']],
                            np.full(sw['diverged'].sum(), 0.96 * y_top))
        if ok.any():
            i = np.flatnonzero(ok)[0]
            self.b_guide.set_data([0, x_max], [0, e[i] / d[i] * x_max])
        else:
            self.b_guide.set_data([], [])
        self.b_msg.set_text('' if ok.any() else
                            'точное решение не дожило\nдо t = %g —\n'
                            'сравнивать не с чем' % T_ERR)

        in_view = np.isfinite(thr) and thr <= x_max
        self.b_thr.set_visible(in_view)
        self.b_thr_txt.set_visible(in_view)
        if in_view:
            self.b_thr.set_xdata([thr, thr])
            self.b_thr_txt.set_x(thr)
        self.b_now.set_visible(dt <= x_max)
        self.b_now.set_xdata([dt, dt])
        self.b_now_txt.set_visible(dt > x_max)
        ax2.set_xlim(0, 1.03 * x_max)
        ax2.set_ylim(0, 1.05 * y_top)
        handles = [h for h in (self.b_sup, self.b_guide, self.b_top, self.b_neg,
                               self.b_div)
                   if np.isfinite(np.asarray(h.get_ydata(), float)).any()]
        if handles:
            ax2.legend(handles=handles, loc='lower right', fontsize=8,
                       framealpha=0.9)
        elif ax2.get_legend() is not None:
            ax2.get_legend().remove()

    # ---------------- числа под графиком ----------------
    def _info_html(self, eq, y0, dt, m, te, ye, sol, t_alive, thr):
        nm, sat = eq.name, self.w_mode.value == MODE_SAT
        if not self.w_eu.value:
            notes = []
            if self.w_iso.value:
                notes.append(self._iso_note(m, sat))
            notes.append("<div style='color:#777;margin-top:4px'>Включите "
                         "«Эйлер» — появятся шаги метода, а справа график "
                         "ошибки.</div>")
            return ("<div style='font-size:13px;line-height:1.6;"
                    "max-width:1120px'>%s</div>" % ''.join(notes))
        fin = np.isfinite(ye)
        first = ' → '.join(_fmt(v) for v in ye[:4][fin[:4]])
        rows = ["Эйлер, первые шаги: %s(0) = %s" % (nm, first)]
        if np.isfinite(thr):
            ratio = dt / thr
            col = 'green' if ratio < 1 else '#b00000'
            rows.append("граница устойчивости в начальной точке ≈ <b>%s</b>, "
                        "шаг <b style='color:%s'>%s</b> её"
                        % (_fmt(thr), col,
                           'в %s раза меньше' % _fmt(1 / ratio, 1) if ratio < 1
                           else 'в %s раза больше' % _fmt(ratio, 1)))
        n_err = int(np.floor(T_ERR / dt + 1e-9))
        notes = []
        if sol is not None and n_err >= 1 and t_alive >= T_ERR - 1e-9:
            k = min(n_err, len(te) - 1)
            e = np.abs(ye[:k + 1] - sol(te[:k + 1])[0])
            if np.all(np.isfinite(e)):
                rows.append("ошибка: наибольшая на [0, %g] — <b>%s</b>, "
                            "в момент t = %g — <b>%s</b>"
                            % (T_ERR, _fmt(e.max(), 3), T_ERR, _fmt(e[-1], 4)))
                if e.max() > 0.3 and e[-1] < 0.1 * e.max():
                    notes.append(
                        "<div style='color:#1f4e9c;margin-top:4px'>На старте "
                        "Эйлер сильно промахнулся, а к t = %g ошибка почти "
                        "исчезла. Это не заслуга метода: все решения "
                        "стягиваются к одному режиму, и он забывает, откуда "
                        "начали, — вместе с ошибкой.</div>" % T_ERR)
        if not np.all(fin):
            t_bad = te[np.argmin(fin)]
            notes.append("<div style='color:#b00000;margin-top:4px'><b>Эйлер "
                         "разлетелся</b> при t ≈ %s: каждый шаг перебрасывает "
                         "решение всё дальше.</div>" % _fmt(t_bad, 2))
        elif eq.physical_positive and np.nanmin(ye) < 0:
            notes.append("<div style='color:#b00000;margin-top:4px'><b>Эйлер "
                         "зашёл в T &lt; 0</b> (минимум %s). Абсолютная "
                         "температура отрицательной не бывает — число на "
                         "экране не температура, а артефакт шага.</div>"
                         % _fmt(np.nanmin(ye), 2))
        if self.w_iso.value:
            notes.append(self._iso_note(m, sat))
        return ("<div style='font-size:13px;line-height:1.6;max-width:1120px'>"
                "%s%s</div>" % (' &nbsp;•&nbsp; '.join(rows), ''.join(notes)))

    def _iso_note(self, m, sat):
        if sat:
            txt = ("Изоклина m = %s: T = ⁴√(1 − m + A·cos t). Вдоль неё "
                   "все чёрточки параллельны — решение, пересекая её, "
                   "идёт с наклоном ровно m. Ниже оси — зеркальная ветвь "
                   "с минусом: «минус не физичен»." % _fmt(m, 1))
            if abs(m - (1 + self.w_A.value)) < 1e-9:
                txt = ("Изоклина m = %s выродилась в точки t = 2πk на оси: "
                       "такой наклон бывает только при T = 0 и cos t = 1."
                       % _fmt(m, 1))
            if m > 1 + self.w_A.value + 1e-9:
                txt = ("Изоклины m = %s нет: наклон не бывает больше "
                       "1 + A = %s, потому что T⁴ ≥ 0."
                       % (_fmt(m, 1), _fmt(1 + self.w_A.value, 2)))
        else:
            txt = ("Изоклина m = %s — линия, где f(t, y) = m. Вдоль неё "
                   "все чёрточки параллельны." % _fmt(m, 1))
        return ("<div style='color:#6f4fb0;margin-top:4px'>%s</div>"
                % txt)

    # ---------------- реакция на события ----------------
    def _refresh(self):
        self.info.value = self.render()
        with self.out:
            clear_output(wait=True)
            display(self.fig)

    def _on_change(self, change=None):
        if not self._busy:
            self._refresh()

    def _set(self, **kw):
        self._busy = True
        try:
            for name, val in kw.items():
                getattr(self, name).value = val
        finally:
            self._busy = False
        self._refresh()


app = SputnikApp()

### Что попробовать: нажимать по порядку

1. **«Изоклины как на доске».** Пунктиром — те же изоклины, что на доске: $m = 1{,}5;\ 1;\ 0;\ -1;\ -15$, снизу вверх. Сплошная с красными чёрточками — одна выделенная; ползунком «изоклина m» можно пройти по всем. Вдоль каждой изоклины чёрточки параллельны — в этом и весь метод. Ниже оси — зеркальная ветвь с минусом перед корнем; «минус не физичен», поэтому область закрашена.

2. **Голубые кривые** — решения из $T(0) = 0{,}3;\ 1;\ 2;\ 4$. Все четыре приходят к одному и тому же периодическому режиму, в том числе то, что стартовало **снизу**, с 0,3. Так что на картинке не «охлаждение», а **притяжение**: с какой температуры ни начни, режим один.

3. **«Эйлер с доски»:** $T(0) = 2$, $\Delta t = 0{,}01$. Под графиком первые шаги: $2 \to 1{,}8600 \to 1{,}7603 \to 1{,}6843$, как на доске. Красная ломаная лежит на синей кривой. Справа появился график ошибки — к нему вернёмся в шаге 6.

4. **«Эйлер врёт»:** $T(0) = 4$, $\Delta t = 0{,}02$. Первый же шаг уводит в $-1{,}08$, **ниже абсолютного нуля**. Посмотрите на поле направлений у $T = 4$: чёрточки почти вертикальны (наклон $-254$), и шаг $0{,}02$ по такому склону пролетает всю картинку. Потом Эйлер выползает обратно и к $t \approx \pi/2$ сливается с режимом.

5. Поднимите шаг до **0,05**: $4 \to -8{,}7 \to -295 \to \ldots$ — разлёт за несколько шагов.

6. **График ошибки справа.** По горизонтали — шаг, по вертикали — наибольшая ошибка Эйлера на отрезке $[0, 2]$. Вернитесь к $T(0) = 2$. На самых малых шагах красная кривая идёт вдоль пунктира: шаг вдвое меньше — ошибка вдвое меньше. Вопрос один: **с какого шага кривая уходит за верхний край, и что при таком шаге происходит на левом графике?** Штрихпунктир — граница устойчивости; почему она стоит именно там, разбирается в задаче Б4 домашнего листа.

7. **«Своё уравнение»** — любое $y' = f(t, y)$, записанное через `t` и `y`: например, `5 - 3*sqrt(y)` или `t - y^2`. Всё то же самое: поле, изоклины, решения, Эйлер, ошибка. Удобно проверить себя после того, как решили руками.

---

## Пикар: почему «при малых t»

На доске вторую итерацию пришлось раскладывать при $t \ll 1$: иначе нужно интегрировать $(1 + \sin s)^4$ — можно, но долго, а третья итерация без разложения уже безнадёжна. Виджет ниже считает итерации честно, численным интегрированием, без всяких разложений, и показывает, **где** они совпадают с решением.

Что видно:

- Каждая итерация отвоёвывает примерно одинаковый кусок времени: $0{,}07$, потом $0{,}14$, $0{,}22$, … — и $0{,}47$ к седьмой. Правее итерации не просто неточны, а разлетаются: их подставляют в $T^4$, и всякая ошибка умножается.
- Галочка **«парабола с доски»** показывает $1 + t - 2t^2$ — то, что получилось на паре. Она совпадает с точным рядом Тейлора решения до $t^2$ включительно и поэтому при малых $t$ даже **ближе** к решению, чем сама $T_2$: при $t = 0{,}1$ парабола даёт $1{,}0800$, $T_2$ — $1{,}0778$, точное решение — $1{,}0809$.

In [ ]:
#@title Виджет: итерации Пикара { display-mode: "form" }
# =============================================================
#  ПИКАР — итерации на отрезке. Требует ячейку с движком.
#  Та же задача, что на доске: A = 1, T(0) = 1.
# =============================================================
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

P_T = np.linspace(0.0, 1.2, 2401)
P_N = 7
P_TOL = 0.01


class PicardApp:

    def __init__(self):
        eq = sputnik(1.0)
        sol, _ = reference(eq, 1.0, P_T[-1])
        self.exact = sol(P_T)[0]
        self.its = picard(eq, 1.0, P_T, P_N)
        self.good_to = []
        for Tn in self.its:
            bad = np.flatnonzero(~(np.abs(Tn - self.exact) < P_TOL))
            self.good_to.append(P_T[bad[0]] if bad.size else P_T[-1])

        self.w_n = widgets.IntSlider(value=2, min=0, max=P_N,
                                     description='итерация n:',
                                     style={'description_width': '90px'},
                                     layout=widgets.Layout(width='320px'))
        self.w_par = widgets.Checkbox(value=False, indent=False,
                                      description='парабола с доски 1 + t − 2t²')
        self.out, self.info = widgets.Output(), widgets.HTML()

        self.fig, self.ax = plt.subplots(figsize=(11.2, 4.2), dpi=88)
        self.fig.subplots_adjust(left=0.06, right=0.985, top=0.9, bottom=0.13)
        plt.close(self.fig)
        ax = self.ax
        self.a_ok = ax.axvspan(0, 0, color='#e3f1e3', zorder=0)
        self.a_prev = [ax.plot([], [], color='#c9b8e6', lw=1.0, zorder=2)[0]
                       for _ in range(P_N)]
        self.a_cur, = ax.plot([], [], color='#6f4fb0', lw=2.4, zorder=4,
                              label='итерация Пикара')
        self.a_ex, = ax.plot(P_T, self.exact, color='black', lw=1.6, zorder=3,
                             label='точное решение')
        self.a_par, = ax.plot([], [], '--', color='#ef8a00', lw=1.8, zorder=5,
                              label='1 + t − 2t² (с доски)')
        ax.set_xlim(P_T[0], P_T[-1])
        ax.set_ylim(0.4, 1.8)
        ax.set_xlabel('t')
        ax.set_ylabel('T')
        ax.grid(True, alpha=0.25)
        ax.legend(loc='lower left', fontsize=9)

        self.w_n.observe(lambda _c: self._refresh(), names='value')
        self.w_par.observe(lambda _c: self._refresh(), names='value')
        display(widgets.HBox([self.w_n, self.w_par]), self.out, self.info)
        self._refresh()

    def _refresh(self):
        n = self.w_n.value
        for k, line in enumerate(self.a_prev):
            if k < n:
                line.set_data(P_T, self.its[k])
            else:
                line.set_data([], [])
        self.a_cur.set_data(P_T, self.its[n])
        self.a_par.set_data((P_T, 1 + P_T - 2 * P_T**2) if self.w_par.value
                            else ([], []))
        t_ok = self.good_to[n]
        self.a_ok.set_x(0)
        self.a_ok.set_width(t_ok)
        self.ax.set_title('T%s(t): совпадает с точным решением с точностью %s '
                          'на [0; %s]' % (str(n).translate(str.maketrans(
                              '0123456789', '₀₁₂₃₄₅₆₇₈₉')),
                              str(P_TOL).replace('.', ','),
                              ('%.3f' % t_ok).replace('.', ',')), fontsize=12)
        steps = ' → '.join(('%.3f' % v).replace('.', ',') for v in self.good_to)
        self.info.value = (
            "<div style='font-size:13px;line-height:1.6'>Отрезок, где итерация "
            "совпадает с точным решением (зелёный), для n = 0, 1, …, %d: %s. "
            "Каждая итерация отвоёвывает примерно одинаковый кусок "
            "времени — и не больше. Правее итерации расходятся с решением "
            "всё сильнее: там их подставляют в T⁴, и ошибка умножается.</div>"
            % (P_N, steps))
        with self.out:
            clear_output(wait=True)
            display(self.fig)


picard_app = PicardApp()

---

## По желанию: средняя температура и $\langle T\rangle_{\text{кв}}$

*Этот раздел — как жёлтая рамка: необязателен и заглядывает в следующее занятие.*

Усреднение уравнения по периоду дало **точное** равенство $\langle T^4 \rangle = Q_0/\sigma$. На доске эта величина обозначена $\langle T\rangle_{\text{кв}} = \sqrt[4]{\langle T^4\rangle}$ — по образцу среднеквадратичной скорости $v_{\text{кв}} = \sqrt{\langle v^2\rangle}$ из молекулярной физики. И так же, как $v_{\text{кв}}$ больше средней скорости $\langle v\rangle$, $\langle T\rangle_{\text{кв}}$ больше средней температуры: корень четвёртой степени из среднего — не среднее.

Ячейка ниже находит установившийся режим численно и считает обе величины, а заодно сравнивает режим с тем, что даст линейная теория на следующем занятии.

In [ ]:
#@title По желанию: средняя температура и линейная теория { display-mode: "form" }
# =============================================================
#  СРЕДНЯЯ ТЕМПЕРАТУРА И ТЕМПЕРАТУРА ПО ИЗЛУЧЕНИЮ.
#  Требует ячейку с движком. Считается секунд десять.
# =============================================================
import matplotlib.pyplot as plt

A_LIST = np.linspace(0.05, 1.0, 20)
regimes = [periodic_regime(A, n=4000) for A in A_LIST]
r1 = periodic_regime(1.0)
lin1 = linear_prediction(1.0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.2, 4.3), dpi=88,
                               gridspec_kw={'width_ratios': [1.5, 1.0]})
fig.subplots_adjust(left=0.06, right=0.985, top=0.88, bottom=0.14, wspace=0.22)

# --- один период при A = 1 ---
deg = np.degrees(OMEGA * r1['t'])
ax1.plot(deg, r1['T'], color='#1f4e9c', lw=2.4, label='T(t), точно')
ax1.plot(deg, lin1['T_star'] + lin1['amp'] * np.cos(np.radians(deg - lin1['phase'])),
         '--', color='#2e8b57', lw=1.6, label='линейная теория (П4)')
ax1.axhline(r1['rad'], color='black', lw=1.2,
            label='⟨T⟩кв = ⁴√⟨T⁴⟩ = ⁴√(Q₀/σ) = %s' % ('%.3f' % r1['rad']).replace('.', ','))
ax1.axhline(r1['mean'], color='#d62728', lw=1.2, ls='-.',
            label='⟨T⟩ = %s' % ('%.3f' % r1['mean']).replace('.', ','))
ax1.plot(deg, 1 + 0.12 * np.cos(np.radians(deg)), ':', color='#ef8a00', lw=1.3,
         label='нагрев (форма, не масштаб)')
for x, c in ((r1['peak'], '#1f4e9c'), (r1['trough'], '#1f4e9c')):
    ax1.axvline(x, color=c, lw=0.8, alpha=0.5)
ax1.set_xlim(0, 360)
ax1.set_xticks(range(0, 361, 45))
ax1.set_xlabel('фаза нагрева ωt, градусы (0 — максимум нагрева)')
ax1.set_ylabel('T')
ax1.set_title('Один период установившегося режима, A = 1', fontsize=12)
ax1.grid(True, alpha=0.25)
ax1.legend(fontsize=8.5, loc='upper right')

# --- от амплитуды нагрева ---
gap = [(r['rad'] - r['mean']) / r['rad'] * 100 for r in regimes]
amp_err = [(linear_prediction(A)['amp'] - r['amp1']) / r['amp1'] * 100
           for A, r in zip(A_LIST, regimes)]
ph_err = [linear_prediction(A)['phase'] - r['phase1']
          for A, r in zip(A_LIST, regimes)]
ax2.plot(A_LIST, gap, '-o', ms=3, color='#d62728',
         label='⟨T⟩кв выше ⟨T⟩, %')
ax2.plot(A_LIST, amp_err, '-o', ms=3, color='#2e8b57',
         label='ошибка линейной теории: амплитуда, %')
ax2.plot(A_LIST, ph_err, '-s', ms=3, color='#6f4fb0',
         label='ошибка линейной теории: запаздывание, градусы')
ax2.axhline(0, color='#888888', lw=0.8)
ax2.set_xlabel('амплитуда нагрева A (при Q₀ = 1)')
ax2.set_title('Чем сильнее качели, тем хуже', fontsize=12)
ax2.grid(True, alpha=0.25)
ax2.legend(fontsize=8.5, loc='lower left')
plt.show()


def _c(x, d):
    return ('%.*f' % (d, x)).replace('.', ',')


print('%5s | %6s %8s | %-17s | %s'
      % ('A', '⟨T⟩', '⟨T⟩кв', 'запаздывание', 'максимум / минимум'))
print('%5s | %6s %8s | %-17s |' % ('', '', '', 'точно / П4'))
print('-' * 66)
for A in (1.0, 0.5, 0.2, 0.05):
    r, l = periodic_regime(A), linear_prediction(A)
    print('%5s | %6s %8s | %-17s | %s° / %s°'
          % (_c(A, 2), _c(r['mean'], 4), _c(r['rad'], 4),
             '%s° / %s°' % (_c(r['phase1'], 1), _c(l['phase'], 1)),
             _c(r['peak'], 1), _c(r['trough'] - 180, 1)))
print('\n«Запаздывание точно» — у синусоиды, которая лучше всего повторяет '
      'настоящую кривую: линейная теория отвечает именно синусоидой.')
print('Последний столбец — на сколько отстают сам максимум T от максимума '
      'нагрева и минимум T от минимума нагрева.')

### Что из этого следует

**Средняя температура ниже, чем $\langle T\rangle_{\text{кв}} = \sqrt[4]{Q_0/\sigma}$.** При $A = 1$ — $0{,}942$ против $1{,}000$, на 6 %. Зазор растёт с размахом нагрева и исчезает, когда нагрев постоянен. Что тогда означает $\sqrt[4]{\langle T^4\rangle}$ и какую из двух температур видно издалека — вопрос В3 домашнего листа.

**Максимум запаздывает мало, минимум — сильно.** Максимум $T$ отстаёт от максимума нагрева на $8{,}7^\circ$, минимум $T$ от минимума нагрева — на $30{,}4^\circ$. Причина в $T^4$: горячий спутник излучает сильно и подстраивается под нагрев быстро (время подстройки $1/(4\sigma T^3) \approx 0{,}15$ при $T = 1{,}19$), холодный излучает слабо и подстраивается медленно ($\approx 1{,}1$ при $T = 0{,}61$). Поэтому кривая — не синусоида: растёт за $158^\circ$ периода, падает за $202^\circ$.

**Насколько окажется верна линейная теория.** На следующем занятии это уравнение линеаризуют и решат точно. Линеаризовать придётся вокруг $\sqrt[4]{Q_0/\sigma} = 1$ — другой температуры до решения мы не знаем. Ответ линейной теории — синусоида, поэтому сравнивать его надо с синусоидой, которая лучше всего повторяет настоящую кривую. При $A = 1$ у линейной теории амплитуда $0{,}243$ вместо $0{,}283$, запаздывание $14{,}0^\circ$ вместо $19{,}0^\circ$ и вдобавок промах по среднему на те же 6 %. При $A = 0{,}2$ расхождение меньше процента и меньше градуса. Линейная теория — теория малых качелей, и граница её применимости видна здесь в цифрах.

---

## Вопросы к виджету

1. **Включите поле направлений и посмотрите на изоклину $m = 0$.** Почему решения, где бы ни начинались, приходят к одному режиму? Что было бы, если бы вместо $-\sigma T^4$ стояло $+\sigma T^4$?

<details><summary>Ответ</summary>

Над изоклиной $m = 0$ наклон отрицательный — решение падает; под ней положительный — растёт. Сама изоклина $m = 0$, кривая $T = \sqrt[4]{Q_0 + A\cos t}$, — это «мгновенное равновесие», и решение всё время за ним гонится: чем дальше от него, тем сильнее тянет обратно.

С $+\sigma T^4$ правая часть $Q_0 + A\cos t + \sigma T^4$ была бы положительна везде (при $Q_0 \ge A$), изоклины $m = 0$ не было бы вовсе. Все решения росли бы, и тем быстрее, чем выше: $T' \ge \sigma T^4$ — это уравнение $f' = f^2$ из первого листа, только круче. Решение уходит на бесконечность за конечное время.

</details>

2. **Поставьте ползунок изоклины на 2 и выше.** Почему изоклина $m = 2$ вырождается в отдельные точки, а изоклин с $m > 2$ нет вовсе?

<details><summary>Ответ</summary>

Потому что $T^4 \ge 0$, и наклон $m = 1 + A\cos t - T^4$ не больше $1 + A = 2$. Равенство — только при $T = 0$ и $\cos t = 1$, то есть в точках $t = 2\pi k$ на оси. Больше 2 наклон не бывает нигде: нагрев ограничен сверху, а излучение может только отнимать.

</details>

3. **«Эйлер врёт»: наибольшая ошибка на $[0, 2]$ — $3{,}47$, а в момент $t = 2$ — всего $0{,}018$.** Значит, метод справился? Проверьте на своём уравнении: сравните ошибку в момент $t = 2$ для `y` и для `-y` при $y(0) = 1$, $\Delta t = 0{,}1$.

<details><summary>Ответ</summary>

Нет. Ошибку исправил не метод, а уравнение: решения притягиваются к одному режиму и забывают начальное условие — вместе с ошибкой в нём.

Где притяжения нет, ошибка копится. Для $y' = y$ при $\Delta t = 0{,}1$ она равна $0{,}12$ при $t = 1$, $0{,}66$ при $t = 2$, $9{,}3$ при $t = 4$. Для $y' = -y$ — $0{,}019$, $0{,}014$, $0{,}0035$: тает. Маленькая ошибка в конце ничего не говорит о точности по дороге.

</details>

4. **Пикар: доведите итерацию до $n = 7$.** Совпадение дотянулось только до $t \approx 0{,}47$. Почему метод, который сходится, продвигается по времени так медленно?

<details><summary>Ответ</summary>

Каждая итерация — интеграл от $0$ до $t$, и под интеграл подставлена предыдущая итерация. Там, где предыдущая уже ошиблась, ошибка попадает в $T^4$ и умножается. Поэтому метод локальный: он хорош около начальной точки и больше нигде не обязан. На доске его по той же причине сразу раскладывали при $t \ll 1$.

Чтобы продвинуться дальше, метод надо перезапускать из новой точки. Если сделать одну итерацию Пикара на коротком отрезке длины $\Delta t$ и интеграл взять по левой точке, получится ровно шаг Эйлера: $T(t + \Delta t) = T(t) + \Delta t\, f(t, T(t))$. Два метода с пары — один и тот же ход в разных масштабах.

</details>

5. *(К разделу «по желанию».)* **Максимум температуры отстаёт от нагрева на $8{,}7^\circ$, минимум — на $30{,}4^\circ$.** Какое из двух чисел — «запаздывание спутника»? Почему линейная теория на следующем занятии даст одно число, а не два?

<details><summary>Ответ</summary>

Оба честные, это разные вещи: кривая несинусоидальна, и её запаздывание зависит от того, по какой точке мерить. Горячий спутник подстраивается быстро, холодный медленно — отсюда разные отставания максимума и минимума.

Линейная теория заменяет $T^4$ касательной прямой, то есть считает скорость подстройки одинаковой при любой температуре. На выходе у неё чистая синусоида и одно запаздывание — $\operatorname{arctg}(\omega/\alpha) = 14{,}0^\circ$ при $\alpha = 4\sigma$. Сравнивать его честно с синусоидой, которая лучше всего повторяет точную кривую (у неё запаздывание $19{,}0^\circ$), а не с максимумом или минимумом.

</details>